In [1]:
import math
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import yfinance as yf

## SPX and SPX Option

In [ ]:
start_date = "2024-06-01"
end_date = "2026-05-01"


API_KEY = os.getenv("MASSIVE_API_KEY") or os.getenv("POLYGON_API_KEY") or os.getenv("Polygon_API_Key")
if not API_KEY:
    raise ValueError("Set MASSIVE_API_KEY (or POLYGON_API_KEY) in your environment.")

# Validate dates
start_dt = pd.to_datetime(start_date)
end_dt = pd.to_datetime(end_date)
if start_dt > end_dt:
    raise ValueError("start_date must be <= end_date")

CONTRACTS_URL = "https://api.massive.com/v3/reference/options/contracts"
HEADERS = {"Authorization": f"Bearer {API_KEY}"}
MAX_WORKERS = 16  # tune based on your API plan/rate limits
RISK_FREE_RATE = 0.04


def is_business_day(date_str: str) -> bool:
    """Check if date is a weekday (basic filter)."""
    ts = pd.Timestamp(date_str)
    return ts.weekday() < 5  # 0-4 = Mon-Fri


def fetch_contracts_for_as_of(as_of_date: str) -> list[dict]:
    """Fetch SPX option contracts for one as_of date with TTM < 6 months."""
    as_of_ts = pd.to_datetime(as_of_date)
    exp_lt = (as_of_ts + pd.Timedelta(days=183)).strftime("%Y-%m-%d")

    params = {
        "underlying_ticker": "SPX",
        "as_of": as_of_date,
        "expiration_date.gte": as_of_date,
        "expiration_date.lt": exp_lt,
        "limit": 1000,
        "sort": "ticker",
        "order": "asc",
    }

    rows = []
    next_url = CONTRACTS_URL

    while next_url:
        response = requests.get(next_url, headers=HEADERS, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()

        page_rows = payload.get("results", [])
        for r in page_rows:
            r["as_of_date"] = as_of_date
        rows.extend(page_rows)

        next_url = payload.get("next_url")
        params = None
        # time.sleep(0.05)

    return rows


def fetch_daily_close_for_ticker(ticker: str, as_of_date: str) -> dict:
    """Fetch daily OHLCV for one option ticker on a specific date."""
    url = f"https://api.massive.com/v2/aggs/ticker/{ticker}/range/1/day/{as_of_date}/{as_of_date}"
    params = {"adjusted": True, "limit": 5}

    try:
        response = requests.get(url, headers=HEADERS, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json()
        results = payload.get("results", [])
        if results:
            bar = results[0]
            return {
                "close": bar.get("c"),
                "open": bar.get("o"),
                "high": bar.get("h"),
                "low": bar.get("l"),
                "volume": bar.get("v"),
            }
    except Exception:
        return {"close": None, "open": None, "high": None, "low": None, "volume": None}

    return {"close": None, "open": None, "high": None, "low": None, "volume": None}


def enrich_rows_with_daily_close(rows: list[dict], as_of_date: str, max_workers: int = MAX_WORKERS) -> list[dict]:
    """Parallelize per-ticker aggs calls to avoid slow sequential requests."""
    if not rows:
        return rows

    ticker_to_indices = {}
    for idx, row in enumerate(rows):
        ticker = row.get("ticker")
        if ticker:
            ticker_to_indices.setdefault(ticker, []).append(idx)

    metrics_map = {}
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        future_map = {
            ex.submit(fetch_daily_close_for_ticker, ticker, as_of_date): ticker
            for ticker in ticker_to_indices
        }
        for future in as_completed(future_map):
            ticker = future_map[future]
            try:
                metrics_map[ticker] = future.result()
            except Exception:
                metrics_map[ticker] = {"close": None, "open": None, "high": None, "low": None, "volume": None}

    for ticker, indices in ticker_to_indices.items():
        m = metrics_map.get(ticker, {})
        for idx in indices:
            rows[idx]["close"] = m.get("close")
            rows[idx]["open"] = m.get("open")
            rows[idx]["high"] = m.get("high")
            rows[idx]["low"] = m.get("low")
            rows[idx]["volume"] = m.get("volume")

    return rows


def fetch_underlying_close(as_of_date: str) -> float | None:
    """Fetch SPX close from yfinance (^SPX)."""
    try:
        start = pd.Timestamp(as_of_date)
        end = start + pd.Timedelta(days=1)
        df_spx = yf.download(
            "^SPX",
            start=start.strftime("%Y-%m-%d"),
            end=end.strftime("%Y-%m-%d"),
            progress=False,
            auto_adjust=False,
        )

        if df_spx.empty:
            return None

        close_col = df_spx.get("Close")
        if close_col is None:
            return None

        last_close = close_col.iloc[-1]
        if isinstance(last_close, pd.Series):
            return float(last_close.iloc[0])

        return float(last_close)
    except Exception:
        return None
    return None


def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def norm_pdf(x: float) -> float:
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)


def bs_price_greeks(S: float, K: float, T: float, r: float, sigma: float, option_type: str) -> dict:
    if S <= 0 or K <= 0 or T <= 0 or sigma <= 0:
        return {"model_price": None, "delta": None, "gamma": None, "theta": None, "vega": None}

    sqrtT = math.sqrt(T)
    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT

    pdf_d1 = norm_pdf(d1)
    cdf_d1 = norm_cdf(d1)
    cdf_d2 = norm_cdf(d2)

    if option_type.lower() == "call":
        price = S * cdf_d1 - K * math.exp(-r * T) * cdf_d2
        delta = cdf_d1
        theta = -S * pdf_d1 * sigma / (2 * sqrtT) - r * K * math.exp(-r * T) * cdf_d2
    else:
        price = K * math.exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)
        delta = cdf_d1 - 1.0
        theta = -S * pdf_d1 * sigma / (2 * sqrtT) + r * K * math.exp(-r * T) * norm_cdf(-d2)

    gamma = pdf_d1 / (S * sigma * sqrtT)
    vega = S * pdf_d1 * sqrtT / 100.0

    return {"model_price": price, "delta": delta, "gamma": gamma, "theta": theta, "vega": vega}


def implied_volatility(price_mkt: float, S: float, K: float, T: float, r: float, option_type: str) -> float | None:
    if price_mkt is None or S is None or K is None or T is None:
        return None
    if S <= 0 or K <= 0 or T <= 0 or price_mkt <= 0:
        return None

    low, high = 1e-4, 5.0
    for _ in range(100):
        mid = 0.5 * (low + high)
        price_mid = bs_price_greeks(S, K, T, r, mid, option_type)["model_price"]
        if price_mid is None:
            return None

        if abs(price_mid - price_mkt) < 1e-5:
            return mid

        if price_mid > price_mkt:
            high = mid
        else:
            low = mid

    return 0.5 * (low + high)


def add_iv_and_greeks(df: pd.DataFrame, r: float) -> pd.DataFrame:
    if df.empty:
        return df

    df = df.copy()
    df["expiration_date"] = pd.to_datetime(df["expiration_date"], errors="coerce")
    df["as_of_date"] = pd.to_datetime(df["as_of_date"], errors="coerce")
    df["days_to_strike"] = (df["expiration_date"] - df["as_of_date"]).dt.days
    df["T"] = df["days_to_strike"] / 252.0

    df["imp_vol"] = None
    df["delta"] = None
    df["gamma"] = None
    df["theta"] = None
    df["vega"] = None

    for idx, row in df.iterrows():
        S = row.get("SPX_close")
        K = row.get("strike_price")
        T = row.get("T")
        px = row.get("close")
        typ = row.get("contract_type")

        iv = implied_volatility(px, S, K, T, r, typ)
        if iv is None:
            continue

        g = bs_price_greeks(S, K, T, r, iv, typ)
        df.at[idx, "imp_vol"] = iv
        df.at[idx, "delta"] = g["delta"]
        df.at[idx, "gamma"] = g["gamma"]
        df.at[idx, "theta"] = g["theta"]
        df.at[idx, "vega"] = g["vega"]

    return df


# ================== MAIN LOOP ==================
all_rows = []
business_days = pd.bdate_range(start_dt, end_dt)   # Only business days

print(f"Processing {len(business_days)} business days from {start_date} to {end_date}\n")

for i, day in enumerate(business_days, start=1):
    as_of = day.strftime("%Y-%m-%d")
    print(f"[{i}/{len(business_days)}] Fetching contracts + daily closes for {as_of} ...")

    try:
        rows = fetch_contracts_for_as_of(as_of)
        rows = enrich_rows_with_daily_close(rows, as_of, max_workers=MAX_WORKERS)

        spx_close = fetch_underlying_close(as_of)
        for r in rows:
            r["SPX_close"] = spx_close

        all_rows.extend(rows)
        print(f"  -> {len(rows)} contracts fetched (close/ohlcv enriched in parallel), SPX close={spx_close}")

    except requests.HTTPError as exc:
        print(f"  !! HTTP error for {as_of}: {exc}")
    except Exception as exc:
        print(f"  !! Error for {as_of}: {exc}")

# ================== SAVE ==================
contracts_df = pd.DataFrame(all_rows)

if not contracts_df.empty:
    # Deduplicate
    contracts_df = contracts_df.drop_duplicates(subset=["ticker", "as_of_date"])

    # Compute implied vol + Greeks using Black-Scholes (r = 0.04)
    contracts_df = add_iv_and_greeks(contracts_df, r=RISK_FREE_RATE)

    # Select columns
    keep_cols = [
        "as_of_date", "ticker", "contract_type", "strike_price", "expiration_date",
        "days_to_strike", "SPX_close", "close", "imp_vol", "delta", "gamma", "theta", "vega",
        "open", "high", "low", "volume"
    ]
    keep_cols = [c for c in keep_cols if c in contracts_df.columns]
    contracts_df = contracts_df[keep_cols]

    output_csv = "spx_options.csv"
    output_parquet = "spx_options.parquet"

    contracts_df.to_csv(output_csv, index=False)
    contracts_df.to_parquet(output_parquet, index=False)

    print(f"\nDone! Saved {len(contracts_df):,} rows")
    print(f"CSV: {output_csv}")
    print(f"Parquet: {output_parquet}")
    print(contracts_df.head())
else:
    print("No data retrieved.")

Processing 2 business days from 2024-06-01 to 2024-06-04

[1/2] Fetching contracts + daily closes for 2024-06-03 ...


KeyboardInterrupt: 

## SPX


In [8]:
spx_start_date = "2024-01-01"
spx_end_date = "2026-05-01"

# yfinance end date is exclusive, so add one day to include spx_end_date
spx_end_exclusive = (pd.Timestamp(spx_end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

spx_data = yf.download(
    "^SPX",
    start=spx_start_date,
    end=spx_end_exclusive,
    progress=False,
    auto_adjust=False,
)

# Single-ticker downloads use a MultiIndex; drop ticker level for a flat CSV
if isinstance(spx_data.columns, pd.MultiIndex):
    spx_data = spx_data.copy()
    spx_data.columns = spx_data.columns.droplevel(1)

spx_data = spx_data.reset_index()
log_ret = np.log(spx_data["Close"] / spx_data["Close"].shift(1))
spx_data["RV22"] = np.sqrt((252.0 / 22.0) * log_ret.pow(2).rolling(22, min_periods=22).sum())
spx_data.to_csv("spx_data.csv", index=False)

print(f"Downloaded {len(spx_data)} SPX (^SPX) rows from {spx_start_date} to {spx_end_date}")
spx_data.head()

Downloaded 585 SPX (^SPX) rows from 2024-01-01 to 2026-05-01


Price,Date,Adj Close,Close,High,Low,Open,Volume,RV22
0,2024-01-02,4742.830078,4742.830078,4754.330078,4722.669922,4745.200195,3743050000,NaN
1,2024-01-03,4704.810059,4704.810059,4729.290039,4699.709961,4725.069824,3950760000,NaN
2,2024-01-04,4688.680176,4688.680176,4726.779785,4687.529785,4697.419922,3715480000,NaN
3,2024-01-05,4697.240234,4697.240234,4721.490234,4682.109863,4690.569824,3844370000,NaN
4,2024-01-08,4763.540039,4763.540039,4764.540039,4699.819824,4703.700195,3742320000,NaN


## VVIX

In [9]:
vvix_start_date = "2024-01-01"
vvix_end_date = "2026-05-01"

# yfinance end date is exclusive
vvix_end_exclusive = (pd.Timestamp(vvix_end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

# CBOE VVIX (volatility of VIX); Yahoo ticker ^VVIX
time.sleep(1)  # small gap helps avoid yfinance rate limits after other download cells

vvix_data = yf.download(
    "^VVIX",
    start=vvix_start_date,
    end=vvix_end_exclusive,
    progress=False,
    auto_adjust=False,
)

if isinstance(vvix_data.columns, pd.MultiIndex):
    vvix_data = vvix_data.copy()
    vvix_data.columns = vvix_data.columns.droplevel(1)

vvix_data = vvix_data.reset_index()
vvix_data.to_csv("vvix_data.csv", index=False)

print(f"Downloaded {len(vvix_data)} VVIX (^VVIX) rows from {vvix_start_date} to {vvix_end_date}")
vvix_data.head()

Downloaded 585 VVIX (^VVIX) rows from 2024-01-01 to 2026-05-01


Price,Date,Adj Close,Close,High,Low,Open,Volume
0,2024-01-02,88.089996,88.089996,92.820000,87.589996,91.860001,0
1,2024-01-03,89.650002,89.650002,92.309998,86.230003,89.260002,0
2,2024-01-04,87.589996,87.589996,90.410004,86.680000,89.919998,0
3,2024-01-05,82.870003,82.870003,86.480003,82.550003,86.459999,0
4,2024-01-08,81.260002,81.260002,85.620003,80.230003,85.620003,0


## VIX

In [10]:
vix_start_date = "2024-01-01"
vix_end_date = "2026-05-01"

# yfinance end date is exclusive
vix_end_exclusive = (pd.Timestamp(vix_end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

# CBOE VIX (volatility of VIX); Yahoo ticker ^VIX
time.sleep(1)  # small gap helps avoid yfinance rate limits after other download cells

vix_data = yf.download(
    "^VIX",
    start=vix_start_date,
    end=vix_end_exclusive,
    progress=False,
    auto_adjust=False,
)

if isinstance(vix_data.columns, pd.MultiIndex):
    vix_data = vix_data.copy()
    vix_data.columns = vix_data.columns.droplevel(1)

vix_data = vix_data.reset_index()
vix_data.to_csv("vix_data.csv", index=False)

print(f"Downloaded {len(vix_data)} VIX (^VIX) rows from {vix_start_date} to {vix_end_date}")
vix_data.head()

Downloaded 585 VIX (^VIX) rows from 2024-01-01 to 2026-05-01


Price,Date,Adj Close,Close,High,Low,Open,Volume
0,2024-01-02,13.20,13.20,14.23,13.10,13.22,0
1,2024-01-03,14.04,14.04,14.22,13.33,13.35,0
2,2024-01-04,14.13,14.13,14.20,13.64,13.93,0
3,2024-01-05,13.35,13.35,14.58,13.29,14.24,0
4,2024-01-08,13.08,13.08,14.18,13.02,14.00,0
